> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — chromosome A/D GRMs and LOCO algebra

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
# Genotypes are simulated once into a 2-bit packed Zarr store (GT_packed2, 0.25 byte per call)
# that every tutorial reuses. Traits are simulated from bounded decoded blocks, so no dense
# animals x variants matrix is held in memory; GENO is the genotype source used below.
DATA=Path("vigor_tutorial_data"); DATA.mkdir(exist_ok=True)
GENO=str(DATA/"complex_n320_m1000_chr5.zarr")
sim=vigor.simulate_packed_cohort(GENO,n_samples=320,n_variants=1000,n_chromosomes=5,n_traits=8,seed=SEED,genotype_kwargs={"generations":100},trait_kwargs=cfg)
Y=sim.phenotypes.to_pandas(); VI=vigor.result_table(sim,"variant_info")
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

rows=[]
# vigor.grm streams one chromosome of the packed store in bounded blocks.
for chrom in sorted(VI.chrom.astype(str).unique()):
    g=vigor.grm(GENO,chromosome=chrom,components=("additive","dominance"),method="vitezica",block_size=512,blas_threads=2)
    rows.append((chrom,int((VI.chrom.astype(str)==chrom).sum()),float(np.mean(np.diag(g["additive"].isel(chromosome=0)))),float(np.mean(np.diag(g["dominance"].isel(chromosome=0))))))
pd.DataFrame(rows,columns=["chromosome","variants","diag_A","diag_D"])


The real-data notebook uses `VigorStore` to persist chromosome GRMs and reconstruct genome-wide/LOCO kernels without re-reading PLINK.

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as td:
    # build_grms reads the packed genotype store directly; no PLINK export is needed.
    store=vigor.open_store(Path(td)/"grms.zarr",mode="a",lazy=False)
    vigor.build_grms(GENO,store=store,components=("additive","dominance"),sample_ids=Y.index,
        fit_mask=fitmask,block_size=256,blas_threads=2,progress=False,persist_genomewide=True)
    chromosome=str(VI.chrom.iloc[0])
    loco=vigor.open_grm(store,component="additive",loco=chromosome,materialize=True)
    assert loco.shape==(len(Y),len(Y))
    print("Cached LOCO:",chromosome,loco.shape)
